# Milestone 2 - Data Wrangling
**Project:** Meeting Transcript Efficiency Analyzer  
**Course:** CAP5771 – Intro to Data Science  

**Inputs:** `data/processed/utterance_level_data.csv`, `data/processed/meeting_level_features.csv`  
**Output:** `data/processed/analysis_ready.csv` — one row per utterance with all engineered features  

Pipeline stages:
1. Load & validate raw processed data
2. Clean & normalize
3. Keyword-based sentence labeling (decision / action / discussion)
4. Feature engineering (redundancy, decision density, action clarity)
5. Validation checks
6. Save analysis-ready dataset

## 1. Load & Validate

In [316]:
import pandas as pd
import numpy as np
import re
import os
from collections import Counter

# Load utterance level data produced in Milestone 1
df = pd.read_csv('data/processed/utterance_level_data.csv')
df_features = pd.read_csv('data/processed/meeting_level_features.csv')

print(f'Utterances loaded : {len(df):,}')
print(f'Meetings loaded   : {df["meeting_id"].nunique()}')
print(f'Columns           : {df.columns.tolist()}')
print('\nDtype summary:')
print(df.dtypes)

Utterances loaded : 134,243
Meetings loaded   : 171
Columns           : ['meeting_id', 'speaker_id', 'begin_time', 'text', 'end_time']

Dtype summary:
meeting_id        str
speaker_id        str
begin_time    float64
text              str
end_time      float64
dtype: object


In [317]:
# Missing value audit
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else '  None — dataset is complete')

# Duplicate utterance check
dups = df.duplicated(subset=['meeting_id', 'speaker_id', 'begin_time', 'text'])
print(f'\nExact duplicate rows: {dups.sum()}')
if dups.any():
    df = df[~dups].reset_index(drop=True)
    print(f'  → Dropped duplicates. Remaining: {len(df):,}')

Missing values per column:
text    1
dtype: int64

Exact duplicate rows: 0


## 2. Clean & Normalize

In [318]:
def clean_text(text: str) -> str:
    """Lowercase, collapse whitespace, strip filler tokens."""
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    # Remove AMI noise tokens like <vocalsound>, <gap>, <disfmarker>
    text = re.sub(r'<[^>]+>', '', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_clean'] = df['text'].apply(clean_text)

# Drop empty utterances that are just noise tokens
empty_mask = df['text_clean'].str.len() == 0
print(f'Empty after cleaning: {empty_mask.sum()} — dropping')
df = df[~empty_mask].reset_index(drop=True)

# Word count on clean text
df['word_count'] = df['text_clean'].str.split().str.len()

print(f'Utterances after cleaning: {len(df):,}')
print(f'Word count range: {df["word_count"].min()} – {df["word_count"].max()}')

Empty after cleaning: 1 — dropping
Utterances after cleaning: 134,242
Word count range: 1 – 101


## 3. Sentence Labeling

Three categories using lexicon matching (no external model required, fully reproducible):  
- **decision** - agreement / conclusion signals  
- **action**  - assignment / next-step signals  
- **discussion** - everything else  

In [319]:
# Keyword lexicons (all lowercase)
DECISION_KW = [
    r'\bwe (decided|agreed|concluded|resolved|confirmed|finalised|finalized)\b',
    r'\b(decision is|so we will|let\'s go with|we\'re going with|we\'ll go with)\b',
    r'\b(agreed|approved|accepted|chosen|settled on|that\'s decided)\b',
]

ACTION_KW = [
    r'\b(action item|follow[- ]?up|next step|todo|to[- ]do)\b',
    r'\b(you will|you should|you need to|you are going to|you\'re going to)\b',
    r'\b(i will|i should|i need to|i\'m going to|i am going to)\b',
    r'\b(can you|could you|please|would you) .{0,30}\b(by|before|until|deadline)\b',
    r'\b(assigned to|responsible for|in charge of|owns|owner)\b',
]

DECISION_PAT = re.compile('|'.join(DECISION_KW))
ACTION_PAT   = re.compile('|'.join(ACTION_KW))

def label_utterance(text: str) -> str:
    if DECISION_PAT.search(text):
        return 'decision'
    if ACTION_PAT.search(text):
        return 'action'
    return 'discussion'

df['label'] = df['text_clean'].apply(label_utterance)

# Summary
label_counts = df['label'].value_counts()
print('Label distribution:')
print(label_counts.to_string())
print(f'\nDecision rate : {label_counts["decision"] / len(df):.2%}')
print(f'Action rate   : {label_counts["action"]   / len(df):.2%}')
print(f'Discussion    : {label_counts["discussion"]/ len(df):.2%}')

Label distribution:
label
discussion    132019
action          2033
decision         190

Decision rate : 0.14%
Action rate   : 1.51%
Discussion    : 98.34%


## 4. Feature Engineering

All three core metrics are computed **per meeting** then merged back to utterance level.

| Feature | Definition |
|---|---|
| `decision_density` | decisions / total utterances |
| `action_density` | action items / total utterances |
| `redundancy_score` | fraction of utterances whose bigrams are ≥30% shared with a prior utterance |
| `action_clarity` | fraction of action utterances containing an explicit owner or deadline word |
| `efficiency_score` | composite: `0.4*decision_density + 0.3*action_density + 0.3*(1-redundancy_score)` |

In [320]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

OWNER_DEADLINE_PAT = re.compile(
    r'\b(i will|you will|he will|she will|they will|'
    r'by (monday|tuesday|wednesday|thursday|friday|next week|eod|tomorrow)|'
    r'deadline|due date|before|assigned to)\b'
)

def compute_redundancy(texts, threshold=0.3):
    """Fraction of utterances that are ≥threshold cosine-similar to any prior utterance."""
    if len(texts) < 2:
        return 0.0
    try:
        vec = CountVectorizer(ngram_range=(2, 2), min_df=1).fit_transform(texts)
    except ValueError:   # all texts too short for bigrams
        return 0.0
    redundant = 0
    for i in range(1, vec.shape[0]):
        sim = cosine_similarity(vec[i], vec[:i]).max()
        if sim >= threshold:
            redundant += 1
    return redundant / len(texts)

def compute_action_clarity(action_texts):
    if len(action_texts) == 0:
        return 0.0
    clear = sum(1 for t in action_texts if OWNER_DEADLINE_PAT.search(t))
    return clear / len(action_texts)

# Per-meeting aggregation
meeting_metrics = []

for mid, grp in df.groupby('meeting_id'):
    total    = len(grp)
    n_dec    = (grp['label'] == 'decision').sum()
    n_act    = (grp['label'] == 'action').sum()
    dec_den  = n_dec / total
    act_den  = n_act / total
    redund   = compute_redundancy(grp['text_clean'].tolist())
    act_cl   = compute_action_clarity(grp.loc[grp['label']=='action', 'text_clean'].tolist())
    eff      = 0.4 * dec_den + 0.3 * act_den + 0.3 * (1 - redund)

    meeting_metrics.append({
        'meeting_id'       : mid,
        'total_utterances' : total,
        'n_decisions'      : n_dec,
        'n_actions'        : n_act,
        'decision_density' : round(dec_den, 4),
        'action_density'   : round(act_den, 4),
        'redundancy_score' : round(redund, 4),
        'action_clarity'   : round(act_cl, 4),
        'efficiency_score' : round(eff, 4),
    })

df_meeting_metrics = pd.DataFrame(meeting_metrics)

print(f'Meeting-level metrics computed for {len(df_meeting_metrics)} meetings')
print(df_meeting_metrics.describe().round(3))

Meeting-level metrics computed for 171 meetings
       total_utterances  n_decisions  n_actions  decision_density  \
count           171.000      171.000    171.000           171.000   
mean            785.041        1.111     11.889             0.001   
std             385.390        1.399      8.714             0.002   
min             113.000        0.000      1.000             0.000   
25%             529.500        0.000      6.000             0.000   
50%             711.000        1.000     10.000             0.001   
75%            1008.000        2.000     15.000             0.002   
max            2419.000        8.000     55.000             0.010   

       action_density  redundancy_score  action_clarity  efficiency_score  
count         171.000           171.000         171.000           171.000  
mean            0.017             0.184           0.241             0.250  
std             0.011             0.062           0.228             0.020  
min             0.002     

In [321]:
# Merge meeting-level metrics back to utterance level
df_analysis = df.merge(df_meeting_metrics, on='meeting_id', how='left')

# Utterance-level positional feature: relative position in meeting (0–1)
df_analysis['utterance_position'] = (
    df_analysis.groupby('meeting_id').cumcount() /
    df_analysis.groupby('meeting_id')['meeting_id'].transform('count')
).round(4)

# Binary label columns for modeling
df_analysis['is_decision'] = (df_analysis['label'] == 'decision').astype(int)
df_analysis['is_action']   = (df_analysis['label'] == 'action').astype(int)

print(f'Analysis-ready dataframe shape: {df_analysis.shape}')
print(df_analysis.head(3).to_string())

Analysis-ready dataframe shape: (134242, 19)
  meeting_id speaker_id   begin_time                                                                                                     text     end_time                                                                                               text_clean  word_count       label  total_utterances  n_decisions  n_actions  decision_density  action_density  redundancy_score  action_clarity  efficiency_score  utterance_position  is_decision  is_action
0    EN2001a     MEO069  3302.969971  IF YOU IF YOU S. S. H. AND THEY HAVE THIS BIG WARNING ABOUT DOING NOTHING AT ALL IN THE GATEWAY MACHINE  3305.969971  if you if you s. s. h. and they have this big warning about doing nothing at all in the gateway machine          22  discussion              1675            1         31            0.0006          0.0185             0.191          0.1935            0.2485              0.0000            0          0
1    EN2001a     MEE068  4149.149902       

## 5. Validation Checks

In [322]:
# Schema validation
required_cols = [
    'meeting_id', 'speaker_id', 'begin_time', 'text_clean',
    'word_count', 'label', 'decision_density', 'action_density',
    'redundancy_score', 'action_clarity', 'efficiency_score',
    'utterance_position', 'is_decision', 'is_action'
]
missing_cols = [c for c in required_cols if c not in df_analysis.columns]
assert not missing_cols, f'Missing columns: {missing_cols}'

# Range checks
for col in ['decision_density','action_density','redundancy_score',
            'action_clarity','efficiency_score','utterance_position']:
    assert df_analysis[col].between(0, 1).all(), f'{col} out of [0,1] range'

# No nulls in key columns
null_check = df_analysis[required_cols].isnull().sum()
assert null_check.sum() == 0, f'Unexpected nulls: {null_check[null_check>0]}'

print('All validation checks passed')
print(f'Final shape: {df_analysis.shape}')

All validation checks passed
Final shape: (134242, 19)


## 6. Save Outputs

In [323]:
os.makedirs('data/processed', exist_ok=True)

# Utterance-level analysis-ready dataset (primary input for modeling)
df_analysis.to_csv('data/processed/analysis_ready.csv', index=False)

# Meeting-level summary (used for visualization dashboard)
df_meeting_metrics.to_csv('data/processed/meeting_metrics.csv', index=False)

print('Saved:')
print(f'  data/processed/analysis_ready.csv       ({len(df_analysis):,} rows)')
print(f'  data/processed/meeting_metrics.csv      ({len(df_meeting_metrics):,} rows)')

Saved:
  data/processed/analysis_ready.csv       (134,242 rows)
  data/processed/meeting_metrics.csv      (171 rows)
